# 06_train_diffusion.ipynb

**目标**: 在 LIBERO 单任务上训练 **Diffusion Policy**，与 03_train_mlp 的
ResNet+MLP baseline 进行对比实验。

**核心区别 vs MLP baseline**:
- 输入: 单帧图像 → 同样单帧图像 (条件)
- 输出: 单步 action (7维) → **action 序列** (horizon=16, 7维)
- 训练目标: MSE(预测动作, 真实动作) → MSE(预测噪声, 真实噪声) (DDPM)
- 推理: 一次前向 → DDIM 多步去噪 (10 步)

**显存约束** (RTX 3070 Ti Laptop, 8GB): batch_size=16, horizon=16


In [1]:
# 导入库
import sys
import importlib.util
from pathlib import Path
import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm

# 动态导入自定义模块
def _load_local(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

dataset_mod = _load_local('dataset02', Path.cwd() / '02_dataset.py')
LIBERODataset = dataset_mod.LIBERODataset

dp_mod = _load_local('diff_policy', Path.cwd() / '05_diffusion_policy.py')
DiffusionPolicy = dp_mod.DiffusionPolicy

from libero.libero import get_libero_path

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


PyTorch: 2.3.1+cu121, CUDA: True
GPU: NVIDIA GeForce RTX 3070 Ti Laptop GPU


## 1. 数据准备 (horizon=16 序列采样)

In [2]:
# 选择第一个任务（与 MLP baseline 完全相同，保证可比性）
dataset_dir = Path(get_libero_path("datasets")) / "libero_spatial"
demo_files = sorted(dataset_dir.glob("*.hdf5"))
task_file = str(demo_files[0])
print(f"Using task: {Path(task_file).name}")

HORIZON = 16   # 动作序列长度（Diffusion Policy 论文标准）

# seq_len=HORIZON 时 Dataset 返回 (T, 3, H, W) 图像和 (T, 7) 动作
dataset = LIBERODataset(
    task_file,
    seq_len=HORIZON,
    image_key="agentview_rgb",
    use_state=False,
)

print(f"Dataset size (seq_len={HORIZON}): {len(dataset)}")

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(
    dataset, [train_size, test_size],
    generator=torch.Generator().manual_seed(42),
)
print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")


Using task: pick_up_the_black_bowl_between_the_plate_and_the_ramekin_and_place_it_on_the_plate_demo.hdf5
Loaded 50 demos from pick_up_the_black_bowl_between_the_plate_and_the_ramekin_and_place_it_on_the_plate_demo.hdf5
  Total valid samples (seq_len=16): 4318
Dataset size (seq_len=16): 4318
Train: 3454, Test: 864


In [3]:
# DataLoader: 显存只有 8GB, batch_size=16 较稳
BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# 检查 batch
sample = next(iter(train_loader))
print(f"images: {sample['images'].shape}  ({sample['images'].dtype})")  # (B, T, 3, 128, 128)
print(f"actions: {sample['actions'].shape} ({sample['actions'].dtype})")  # (B, T, 7)
print(f"action range: [{sample['actions'].min():.3f}, {sample['actions'].max():.3f}]")


images: torch.Size([16, 16, 3, 128, 128])  (torch.float32)
actions: torch.Size([16, 16, 7]) (torch.float32)
action range: [-1.000, 1.000]


## 2. 模型: Diffusion Policy (ResNet18 + 1D Conditional U-Net)

`compute_loss(images, actions)` 内部:
1. 随机采样 timestep $t \sim U(0, T)$
2. 给 clean action 加噪: $a_t = \sqrt{\bar\alpha_t}\,a_0 + \sqrt{1-\bar\alpha_t}\,\epsilon$
3. UNet 预测噪声 $\hat\epsilon = f_\theta(a_t, t, \text{img\_feat})$
4. Loss $= \text{MSE}(\hat\epsilon, \epsilon)$


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

policy = DiffusionPolicy(
    action_dim=7,
    horizon=HORIZON,
    cond_dim=256,
    num_train_timesteps=100,   # 训练去噪步数
    num_inference_steps=10,    # DDIM 推理步数（10 步即可）
).to(device)

n_params = sum(p.numel() for p in policy.parameters())
print(f"Diffusion Policy params: {n_params / 1e6:.2f}M")
print(f"Device: {device}")

# 烟雾测试: forward + loss 能跑
with torch.no_grad():
    sample = next(iter(train_loader))
    imgs = sample['images'].to(device)
    acts = sample['actions'].to(device)
    print(f"images: {imgs.shape}, actions: {acts.shape}")
loss = policy.compute_loss(imgs, acts)
print(f"Initial loss (random init): {loss.item():.4f}")


/home/dongniuniu/miniconda3/envs/libero/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/dongniuniu/miniconda3/envs/libero/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Diffusion Policy params: 15.16M
Device: cuda
images: torch.Size([16, 16, 3, 128, 128]), actions: torch.Size([16, 16, 7])
Initial loss (random init): 1.1200


## 3. 训练配置 (使用 EMA 提升采样稳定性)

In [5]:
import copy

# Diffusion Policy 论文用 AdamW + cosine schedule
optimizer = optim.AdamW(policy.parameters(), lr=1e-4, weight_decay=1e-6)

NUM_EPOCHS = 50
EVAL_EVERY = 5
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# EMA: 对 diffusion 模型很重要，平滑权重，提升采样质量
class EMAModel:
    def __init__(self, model, decay=0.995):
        self.decay = decay
        self.shadow = copy.deepcopy(model).eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for ema_p, p in zip(self.shadow.parameters(), model.parameters()):
            ema_p.data.mul_(self.decay).add_(p.data, alpha=1 - self.decay)
        # buffers 直接复制
        for ema_b, b in zip(self.shadow.buffers(), model.buffers()):
            ema_b.data.copy_(b.data)

ema = EMAModel(policy, decay=0.995)

history = {'train_loss': [], 'test_loss': [], 'lr': []}

print(f"Optimizer: AdamW (lr=1e-4)")
print(f"Epochs: {NUM_EPOCHS}, batch_size: {BATCH_SIZE}")
print(f"EMA decay: 0.995")


Optimizer: AdamW (lr=1e-4)
Epochs: 50, batch_size: 16
EMA decay: 0.995


## 4. 训练循环

In [6]:
def train_epoch(policy, loader, optimizer, ema, device):
    policy.train()
    total = 0.0
    n = 0
    for batch in tqdm(loader, desc="Train", leave=False):
        imgs = batch['images'].to(device)     # (B, T, 3, H, W)
        acts = batch['actions'].to(device)    # (B, T, 7)

        optimizer.zero_grad()
        loss = policy.compute_loss(imgs, acts)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        optimizer.step()
        ema.update(policy)

        bs = imgs.shape[0]
        total += loss.item() * bs
        n += bs
    return total / n


@torch.no_grad()
def eval_epoch(policy, loader, device):
    policy.eval()
    total = 0.0
    n = 0
    for batch in tqdm(loader, desc="Eval", leave=False):
        imgs = batch['images'].to(device)
        acts = batch['actions'].to(device)
        loss = policy.compute_loss(imgs, acts)
        bs = imgs.shape[0]
        total += loss.item() * bs
        n += bs
    return total / n


print("Starting Diffusion Policy training...\n")

for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch(policy, train_loader, optimizer, ema, device)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['lr'].append(scheduler.get_last_lr()[0])

    if (epoch + 1) % EVAL_EVERY == 0:
        # 在 EMA 模型上评估（这是论文推荐做法）
        test_loss = eval_epoch(ema.shadow, test_loader, device)
        history['test_loss'].append(test_loss)
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | Train: {train_loss:.6f} | Test(EMA): {test_loss:.6f}")
    else:
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | Train: {train_loss:.6f}")

print("\nTraining done!")


Starting Diffusion Policy training...



Epoch   1/50 | Train: 0.658636


Epoch   2/50 | Train: 0.352986


Epoch   3/50 | Train: 0.253481


Epoch   4/50 | Train: 0.213828


...

Epoch  46/50 | Train: 0.054904


Epoch  47/50 | Train: 0.054975


Epoch  48/50 | Train: 0.056855


Epoch  49/50 | Train: 0.054961


Epoch  50/50 | Train: 0.054288 | Test(EMA): 0.064581

Training done!


## 5. 保存模型 + 损失曲线

In [7]:
# 保存：原模型 + EMA 模型 + history
save_path = Path('outputs') / 'model_diffusion_policy.pt'
save_path.parent.mkdir(exist_ok=True)

torch.save({
    'model_state_dict': policy.state_dict(),
    'ema_state_dict': ema.shadow.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'history': history,
    'config': {
        'action_dim': 7,
        'horizon': HORIZON,
        'num_train_timesteps': 100,
        'num_inference_steps': 10,
    },
}, save_path)
print(f"Saved to {save_path}  ({save_path.stat().st_size / 1e6:.1f} MB)")


Saved to outputs/model_diffusion_policy.pt  (243.0 MB)


In [8]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# 如果 history 不在内存里则从 checkpoint 加载
if 'history' not in dir() or not history['train_loss']:
    ckpt = torch.load('outputs/model_diffusion_policy.pt', map_location='cpu')
    history = ckpt['history']
    EVAL_EVERY = 5

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Train Loss (noise MSE)')
test_x = [i * EVAL_EVERY + (EVAL_EVERY - 1) for i in range(len(history['test_loss']))]
axes[0].plot(test_x, history['test_loss'], 'o-', label='Test Loss (EMA)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')
axes[0].set_title('Diffusion Policy Training')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history['lr'])
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('LR')
axes[1].set_title('Learning Rate'); axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
Path('outputs').mkdir(exist_ok=True)
plt.savefig('outputs/diffusion_training_curves.png', dpi=100, bbox_inches='tight')
plt.close()

print("=== Diffusion Policy Summary ===")
print(f"Initial train loss: {history['train_loss'][0]:.6f}")
print(f"Final train loss:   {history['train_loss'][-1]:.6f}")
print(f"Loss reduction:     {(1 - history['train_loss'][-1]/history['train_loss'][0])*100:.1f}%")
if history['test_loss']:
    print(f"Final test loss:    {history['test_loss'][-1]:.6f}")
print("Saved curves to outputs/diffusion_training_curves.png")


=== Diffusion Policy Summary ===
Initial train loss: 0.658636
Final train loss:   0.054288
Loss reduction:     91.8%
Final test loss:    0.064581
Saved curves to outputs/diffusion_training_curves.png


## 6. 推理测试: DDIM 采样动作序列

In [9]:
# 用 EMA 模型采样
ema.shadow.eval()
batch = next(iter(test_loader))
imgs = batch['images'].to(device)
acts_gt = batch['actions'].to(device)   # (B, T, 7)

import time
t0 = time.time()
sampled = ema.shadow.sample(imgs, num_inference_steps=10)  # (B, T, 7)
print(f"DDIM 10-step sampling time: {time.time()-t0:.2f}s for batch of {imgs.shape[0]}")

# 第一帧 action 的预测误差（最关心的，因为执行时只用第一步）
err_first = torch.abs(sampled[:, 0] - acts_gt[:, 0]).mean().item()
err_full = torch.abs(sampled - acts_gt).mean().item()
print(f"\nMean abs error - first step: {err_first:.4f}")
print(f"Mean abs error - full horizon: {err_full:.4f}")

print("\n=== Sample 0 ===")
print("GT action[0:3]:")
print(acts_gt[0, :3].cpu().numpy())
print("Pred action[0:3]:")
print(sampled[0, :3].cpu().numpy())


DDIM 10-step sampling time: 0.14s for batch of 16

Mean abs error - first step: 0.2303
Mean abs error - full horizon: 0.1956

=== Sample 0 ===
GT action[0:3]:
[[ 0.9375      0.23035714 -0.3375      0.          0.         -0.
  -1.        ]
 [ 0.9375      0.15535714 -0.42857143  0.01821429  0.         -0.00535714
  -1.        ]
 [ 0.88125     0.11785714 -0.55714285  0.06964286 -0.0075     -0.015
  -1.        ]]
Pred action[0:3]:
[[ 0.9927229   0.13780965 -0.50886065 -0.16962145 -0.1309508  -0.03206012
  -0.9907876 ]
 [ 0.6831914   0.1046638  -0.5352615   0.0069503  -0.12886225 -0.14551957
  -1.0299383 ]
 [ 0.6597463   0.21645296 -0.48663747  0.03176877 -0.08055154  0.10968201
  -0.7229965 ]]
